In [30]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
import os
import json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

In [48]:
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")

# ID mappings (from 02_id_encoder.ipynb)
id_mappings_dir = os.path.join(preprocessed_dir, "id_mappings")
user_id_2idx_path = os.path.join(id_mappings_dir, "user_id_2idx.json")
item_id_2idx_path = os.path.join(id_mappings_dir, "item_id_2idx.json")
scene_2idx_path = os.path.join(id_mappings_dir, "scene_2idx.json")
id_vocab_config_path = os.path.join(id_mappings_dir, "id_vocab_config.json")

# Precomputed embeddings
text_embeddings_path = os.path.join(preprocessed_dir, "text_embeddings.npy")               # (N_items, 768)
image_embeddings_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")
time_features_path = os.path.join(preprocessed_dir, "time_features_final.pt")              # (N_interactions, D_time)
time_feature_cols_path = os.path.join(preprocessed_dir, "time_features_columns.json")      # list of column names

# fusion artifacts from this notebook will be saved here
fusion_artifacts_dir = os.path.join(base_dir, "fusion_artifacts")
os.makedirs(fusion_artifacts_dir, exist_ok=True)

In [49]:
# sanity check
print("base_dir:", base_dir)
print("preprocessed_dir:", preprocessed_dir)
print("id_mappings_dir:", id_mappings_dir)
print("text_embeddings_path:", text_embeddings_path)
print("image_embeddings_path:", image_embeddings_path)
print("time_features_path:", time_features_path)
print("fusion_artifacts_dir:", fusion_artifacts_dir)

base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
id_mappings_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/id_mappings
text_embeddings_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/text_embeddings.npy
image_embeddings_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/image_embeddings_vit_base_aligned.pt
time_features_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/time_features_final.pt
fusion_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts


In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [50]:
# Next, we will load Embedding Sizes (Dynamic Sizing)
# Load shapes without pulling everything into GPU memory
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")
text_dim = text_embeddings.shape[1]

image_embeddings = torch.load(image_embeddings_path, map_location="cpu")
image_dim = image_embeddings.shape[1]

time_features = torch.load(time_features_path, map_location="cpu")
time_raw_dim = time_features.shape[1]

In [51]:
# Load time feature names
with open(time_feature_cols_path, "r") as f:
    time_feature_cols = json.load(f)

print("\n--- Embedding/Feature Dimensions ---")
print("Text embedding dim:", text_dim)
print("Image embedding dim:", image_dim)
print("Time raw feature dim:", time_raw_dim)
print("Time feature columns:", time_feature_cols)


--- Embedding/Feature Dimensions ---
Text embedding dim: 1536
Image embedding dim: 768
Time raw feature dim: 5
Time feature columns: ['unix_time_norm', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend']


In [52]:
# Load vocab sizes (for ID embeddings)
with open(id_vocab_config_path, "r") as f:
    id_vocab_config = json.load(f)

user_vocab_size = int(id_vocab_config["num_users"])
item_vocab_size = int(id_vocab_config["num_items"])
scene_vocab_size = int(id_vocab_config["num_scenes"])

print("\n--- ID Vocabulary Sizes ---")
print("user_vocab_size:", user_vocab_size)
print("item_vocab_size:", item_vocab_size)
print("scene_vocab_size:", scene_vocab_size)


--- ID Vocabulary Sizes ---
user_vocab_size: 67615
item_vocab_size: 59142
scene_vocab_size: 5


## Load All Precomputed Embeddings / Feature Shapes

In this step, we validate and summarize the shapes of all modality inputs that will enter the ColdFusionX fusion module.
These inputs reference back to earlier notebooks:
*   Text embeddings from 03_text_encoder_BERT.ipynb
*   Image embeddings from 04_image_encoder.ipynb (ViT-base CLS, 768-d)
*   Time features from 05_time_encoder.ipynb (raw time vector, 5-d)
*   ID vocab sizes from 02_id_encoder.ipynb (user/item/scene mappings)

To keep memory usage low, we first load arrays/tensors in a lightweight way (e.g., NumPy memmap, CPU tensors).
We then run sanity checks to ensure dimensions match expected vocab sizes and that indexing will remain consistent during model training.

In [53]:
# Text embeddings (NumPy memmap → doesn't fully load into RAM)
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")
num_text_rows, text_dim = text_embeddings.shape

# Image embeddings (torch tensor on CPU)
image_embeddings = torch.load(image_embeddings_path, map_location="cpu")
num_image_rows, image_dim = image_embeddings.shape

# Time features (torch tensor on CPU)
time_features = torch.load(time_features_path, map_location="cpu")
num_time_rows, time_raw_dim = time_features.shape

In [54]:
print("------ Precomputed Embeddings / Feature Shapes ------")
print(f"Text embeddings:  shape = ({num_text_rows}, {text_dim})")
print(f"Image embeddings: shape = ({num_image_rows}, {image_dim})")
print(f"Time features:    shape = ({num_time_rows}, {time_raw_dim})")

print("\n------ Expected Vocab / Cardinalities ------")
print(f"Item vocab size (from id_vocab_config): {item_vocab_size}")
print(f"Scene vocab size (from id_vocab_config): {scene_vocab_size}")
print(f"User vocab size (from id_vocab_config): {user_vocab_size}")

------ Precomputed Embeddings / Feature Shapes ------
Text embeddings:  shape = (3730329, 1536)
Image embeddings: shape = (59142, 768)
Time features:    shape = (3730329, 5)

------ Expected Vocab / Cardinalities ------
Item vocab size (from id_vocab_config): 59142
Scene vocab size (from id_vocab_config): 5
User vocab size (from id_vocab_config): 67615


In [56]:
# Interaction-level alignment (text + time)
assert num_text_rows == num_time_rows, (
    f"Mismatch: text rows ({num_text_rows}) != time rows ({num_time_rows})"
)

# Time feature dimension alignment
assert time_raw_dim == len(time_feature_cols), (
    f"Mismatch: time_raw_dim ({time_raw_dim}) != number of time cols ({len(time_feature_cols)})"
)

# Image coverage check (item-level, may be < item_vocab_size)
missing_images = item_vocab_size - num_image_rows
print(f"Image coverage: {num_image_rows}/{item_vocab_size} items have embeddings.")
print(f"Items missing image embeddings: {missing_images}")

assert missing_images >= 0, "Unexpected: more image embeddings than items."

print("Interaction-level checks passed (text + time). Image coverage reported above.")

Image coverage: 59142/59142 items have embeddings.
Items missing image embeddings: 0
Interaction-level checks passed (text + time). Image coverage reported above.


In [42]:
# Item vocab (all known items)
with open(item_id_2idx_path, "r") as f:
    item_id_2idx = json.load(f)

all_vocab_item_ids = set(int(k) for k in item_id_2idx.keys())

# Item ids that got image embeddings (saved from image encoder)
# (Adjust path if your mapping is saved elsewhere)
image_item_ids_path = f"{base_dir}/item_id_to_index.json"
with open(image_item_ids_path, "r") as f:
    embedded_item_ids = json.load(f)

embedded_item_ids = set(int(x) for x in embedded_item_ids)

missing_item_ids = sorted(all_vocab_item_ids - embedded_item_ids)
print("Missing item_ids (no image embedding):", missing_item_ids)
print("Count:", len(missing_item_ids))

Missing item_ids (no image embedding): [211034, 476407]
Count: 2


In [45]:
# Paths
image_item_ids_path = os.path.join(base_dir, "item_id_to_index.json")  # from 04_image_encoder.ipynb

aligned_image_embeddings_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")

print("item_id_2idx_path:", item_id_2idx_path)
print("image_embeddings_path:", image_embeddings_path)
print("image_item_ids_path:", image_item_ids_path)

# Load mappings
with open(item_id_2idx_path, "r") as f:
    item_id_2idx = json.load(f)

# vocab indices (item_idx)
# item_id_2idx keys are item_id strings -> indices are ints
num_items = len(item_id_2idx)
print("num_items (vocab):", num_items)

# Load embeddings (as extracted)
img_emb = torch.load(image_embeddings_path, map_location="cpu")  # (N_embedded, 768)
N_embedded, emb_dim = img_emb.shape
print("raw image embedding matrix:", img_emb.shape)

# Load the ordered list of item_ids used when embeddings were created
with open(image_item_ids_path, "r") as f:
    embedded_item_ids_list = json.load(f)

assert len(embedded_item_ids_list) == N_embedded, (
    f"Mismatch: item_id list ({len(embedded_item_ids_list)}) != embedding rows ({N_embedded})"
)

# Create aligned matrix
aligned = torch.zeros((num_items, emb_dim), dtype=img_emb.dtype)   # (num_items, 768)
missing_mask = torch.ones((num_items,), dtype=torch.bool)          # True = missing by default

filled = 0
skipped_not_in_vocab = 0

for row_idx, item_id in enumerate(embedded_item_ids_list):
    item_id_int = int(item_id)

    # Only align if item_id exists in vocab
    if str(item_id_int) not in item_id_2idx:
        skipped_not_in_vocab += 1
        continue

    item_idx = int(item_id_2idx[str(item_id_int)])
    aligned[item_idx] = img_emb[row_idx]
    missing_mask[item_idx] = False
    filled += 1

print("\n--- Alignment Summary ---")
print("Filled embeddings:", filled)
print("Skipped (not in vocab):", skipped_not_in_vocab)
print("Missing in aligned matrix:", missing_mask.sum().item())
print("Aligned matrix shape:", aligned.shape)

item_id_2idx_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/id_mappings/item_id_2idx.json
image_embeddings_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/image_embeddings_vit_base.pt
image_item_ids_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/item_id_to_index.json
num_items (vocab): 59142
raw image embedding matrix: torch.Size([59140, 768])

--- Alignment Summary ---
Filled embeddings: 59140
Skipped (not in vocab): 0
Missing in aligned matrix: 2
Aligned matrix shape: torch.Size([59142, 768])


In [46]:
# Save
torch.save(aligned, aligned_image_embeddings_path)
torch.save(missing_mask, missing_mask_path)

print("\nSaved aligned embeddings to:", aligned_image_embeddings_path)
print("Saved missing mask to:", missing_mask_path)


Saved aligned embeddings to: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/image_embeddings_vit_base_aligned.pt
Saved missing mask to: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/image_missing_mask.pt


In [47]:
aligned = torch.load(os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt"), map_location="cpu")
missing_mask = torch.load(os.path.join(base_dir, "image_missing_mask.pt"), map_location="cpu")

print("Aligned:", aligned.shape)
print("Missing count:", missing_mask.sum().item())

# Check that missing rows are actually zero vectors
missing_indices = torch.where(missing_mask)[0]
if len(missing_indices) > 0:
    idx0 = missing_indices[0].item()
    print("First missing idx:", idx0)
    print("Vector norm (should be 0):", torch.norm(aligned[idx0]).item())

Aligned: torch.Size([59142, 768])
Missing count: 2
First missing idx: 9983
Vector norm (should be 0): 0.0


In [57]:
!git commit -m "fixed #3 : align image embeddings with item vocabulary to handle missing/corrupted images"

On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/06_fusion_module.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
